# Arez — Devil's Advocate
**Model:** Mistral Small 3.1 24B Instruct (4-bit quantized)  
**Hardware:** Colab Pro + A100 (40GB VRAM)

### Cách dùng:
1. Chạy **Cell 1** một lần (cài thư viện)
2. Chạy **Cell 2** một lần (load model — ~5 phút)
3. Mỗi lần muốn hỏi: sửa biến `USER_INPUT` trong **Cell 3** rồi Run
4. Muốn reset lịch sử: chạy **Cell 2b**

---

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN  |  Chạy 1 lần duy nhất
# ═══════════════════════════════════════════════════════

import subprocess, sys

pkgs = [
    'transformers>=4.40.0',
    'bitsandbytes>=0.43.0',
    'accelerate>=0.27.0',
    'huggingface_hub',
]

for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ Cài xong.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2 — LOAD MODEL  |  Chạy 1 lần duy nhất (~5 phút)
# ═══════════════════════════════════════════════════════

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'mistralai/Mistral-Small-3.1-24B-Instruct-2503'

# Kiểm tra GPU
assert torch.cuda.is_available(), 'Không tìm thấy GPU. Vào Runtime > Change runtime type > A100'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print(f'Đang load {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.eval()

# System prompt
SYSTEM_PROMPT = """Bạn là Arez — một AI đóng vai Devil's Advocate chuyên nghiệp.

Nhiệm vụ:
- Phản biện mọi luận điểm, quyết định, kế hoạch mà người dùng đưa ra
- Tìm ra điểm yếu, rủi ro ẩn, giả định sai, góc nhìn bị bỏ sót
- Đặt câu hỏi sắc bén, trực tiếp
- KHÔNG đồng ý dễ dàng — luôn challenge
- Trả lời tiếng Việt, ngắn gọn, đi thẳng vào vấn đề

Vai trò là làm cho lập luận của người dùng phải vững hơn — không phải tìm đồng thuận."""

# Khởi tạo lịch sử hội thoại
history = [{'role': 'system', 'content': SYSTEM_PROMPT}]

print('✅ Model sẵn sàng. Chạy Cell 3 để bắt đầu.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 2b — RESET LỊCH SỬ  |  Chạy khi muốn bắt đầu lại
# ═══════════════════════════════════════════════════════

history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
print('🔄 Lịch sử đã reset.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CELL 3 — CHAT  |  Sửa USER_INPUT rồi Run mỗi lần hỏi
# ═══════════════════════════════════════════════════════

USER_INPUT = """Nhập câu hỏi hoặc luận điểm của anh ở đây."""

# ── Không cần chỉnh gì bên dưới ──────────────────────

def ask(user_text, max_new_tokens=512):
    history.append({'role': 'user', 'content': user_text})

    text = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    history.append({'role': 'assistant', 'content': response})
    return response


# Chạy
print(f'Anh: {USER_INPUT}')
print()
print('Arez:', ask(USER_INPUT))